# Cafe Sales — Cleaning + KPI Pipeline
Source: kagglehub `ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training`

Known dirty-data patterns in this file (confirmed from source sample):
- Numeric columns (`Quantity`, `Price Per Unit`, `Total Spent`) contain the literal string `"ERROR"` and true nulls.
- Categorical columns (`Payment Method`, `Location`) contain the literal string `"UNKNOWN"` and true nulls.
- `Transaction Date` contains unparseable / malformed strings.

This script does NOT assume the row count, null rate, or specific corrupt values beyond what's visible —
those are computed from the data at runtime, not hardcoded.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

## 1. Config — adjust the path to wherever the CSV lands in your workspace

In [0]:
INPUT_PATH = "/Workspace/Repos/Git/data-engineering-learnings/session-04: streaming exercise/dirty_cafe_sales.csv"

raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)  # read everything as string first; dirty data breaks type inference
    .csv(INPUT_PATH)
)

display(raw)
# display raw schema
raw.printSchema()

## 2. Normalize sentinel garbage → real nulls
`"ERROR"`, `"UNKNOWN"`, `"NaN"`, `"nan"`, `""`, and whitespace-only strings all become NULL.
This is applied uniformly across every column before any type casting.

In [0]:
# Values that should be treated as NULL
SENTINELS = ["ERROR", "UNKNOWN", "NaN", "nan", "", "NULL", "null", "None"]

cleaned = raw

# Convert garbage values to NULL
for c in raw.columns:
    cleaned = cleaned.withColumn(
        c,
        F.when(
            F.col(c).isNull() | F.trim(F.col(c)).isin(SENTINELS),
            None
        ).otherwise(F.trim(F.col(c)))
    )


cleaned = (
    cleaned
    .withColumn("Quantity", F.col("Quantity").cast("double"))
    .withColumn("Price Per Unit", F.col("Price Per Unit").cast("double"))
    .withColumn("Total Spent", F.col("Total Spent").cast("double"))
)

##3. Recalculate Missing Revenue
If:
Quantity = 2
Price Per Unit = 5
Total Spent = NULL....
we can calculate: 2 × 5 = 10

In [0]:
cleaned = cleaned.withColumn(
    "revenue_was_imputed",
    F.col("Total Spent").isNull()
    & F.col("Quantity").isNotNull()
    & F.col("Price Per Unit").isNotNull()
)

cleaned = cleaned.withColumn(
    "Total Spent",
    F.when(
        F.col("Total Spent").isNull()
        & F.col("Quantity").isNotNull()
        & F.col("Price Per Unit").isNotNull(),
        F.round(
            F.col("Quantity") * F.col("Price Per Unit"),
            2
        )
    ).otherwise(F.col("Total Spent"))
)

##4. Parse Transaction Date
Spark's `dayofweek`: 1 = Sunday, 7 = Saturday. Weekend = Saturday/Sunday.

In [0]:
cleaned = cleaned.withColumn(
    "Transaction_Date_parsed",
    F.to_date(
        F.col("Transaction Date"),
        "yyyy-MM-dd"
    )
)

cleaned = (
    cleaned
    .withColumn(
        "is_invalid_date",
        F.col("Transaction Date").isNotNull()
        & F.col("Transaction_Date_parsed").isNull()
    )
    .withColumn(
        "day_of_week",
        F.dayofweek("Transaction_Date_parsed")
    )
    .withColumn(
        "day_type",
        F.when(
            F.dayofweek("Transaction_Date_parsed").isin(1, 7),
            "Weekend"
        ).otherwise("Weekday")
    )
     .withColumn(
        "date_is_missing",
        F.col("Transaction Date").isNull()
    )
   
)

##5. Create Transaction Success Flag

We'll use your original definition, but keep it simple.
A transaction is successful when we have:
Quantity, Revenue, Payment Method, Location, Valid date

In [0]:
cleaned = cleaned.withColumn(
    "transaction_successful",
    F.col("Quantity").isNotNull()
    & F.col("Total Spent").isNotNull()
    & F.col("Payment Method").isNotNull()
    & F.col("Location").isNotNull()
    & F.col("Transaction_Date_parsed").isNotNull()
)

TOTAL_ROWS = cleaned.count()

## 5. KPI 1 — Revenue by Product

In [0]:
revenue_by_product = (
    cleaned
    .groupBy("Item")
    .agg(
        F.round(F.sum("Total Spent"), 2).alias("Revenue")
    )
    .orderBy(F.desc("Revenue"))
)

display(revenue_by_product)

## 7. KPI 2 — Revenue by Location

In [0]:
revenue_by_location = (
    cleaned
    .groupBy("Location")
    .agg(
        F.round(F.sum("Total Spent"), 2).alias("Revenue")
    )
    .orderBy(F.desc("Revenue"))
)

display(revenue_by_location)

## 8. KPI 3 — Revenue by Payment Method

In [0]:
revenue_by_payment = (
    cleaned
    .groupBy("Payment Method")
    .agg(
        F.round(F.sum("Total Spent"), 2).alias("Revenue")
    )
    .orderBy(F.desc("Revenue"))
)

display(revenue_by_payment)

## 8. KPI 4 — Weekend vs Weekday Sales

In [0]:
weekend_vs_weekday = (
    cleaned
    .groupBy("day_type")
    .agg(
        F.round(F.sum("Total Spent"), 2).alias("Revenue"),
        F.count("*").alias("Transactions")
    )
    .orderBy("day_type")
)

display(weekend_vs_weekday)

## 10. KPI 5 — Peak Sales Day
Single calendar date with the highest total revenue. (If you actually meant "which day of week
peaks", swap groupBy to `dow`/`day_type` from the cell above.)

In [0]:
peak_sales_day = (
    cleaned
    .filter(F.col("Total Spent").isNotNull() & F.col("Transaction_Date_parsed").isNotNull())
    .groupBy("Transaction_Date_parsed")
    .agg(F.round(F.sum("Total Spent"), 2).alias("Revenue"))
    .orderBy(F.desc("Revenue"))
)
display(peak_sales_day.limit(10))
print("Peak day:", peak_sales_day.first())

## 11. KPI 6 — Best Selling Item (by Quantity)

In [0]:
best_selling_item = (
    cleaned
    .groupBy("Item")
    .agg(
        F.sum("Quantity").alias("Quantity_Sold")
    )
    .orderBy(F.desc("Quantity_Sold"))
)

display(best_selling_item)
print("Best seller by quantity:", best_selling_item.first())

## 12. KPI 7 — Product Revenue Contribution (%)

In [0]:
total_revenue = (
    cleaned
    .agg(F.sum("Total Spent").alias("total_revenue"))
    .collect()[0]["total_revenue"]
)

product_contribution = (
    cleaned
    .groupBy("Item")
    .agg(
        F.sum("Total Spent").alias("Revenue")
    )
    .withColumn(
        "Revenue_Contribution_Percent",
        F.round(
            F.col("Revenue") / F.lit(total_revenue) * 100,
            2
        )
    )
    .orderBy(F.desc("Revenue"))
)

display(product_contribution)

## 13. KPI 8 — Transaction Success Rate
% of rows that are "successful" per the row_successful definition in Step 5.

In [0]:
total_transactions = cleaned.count()
successful_transactions = (
    cleaned
    .filter(F.col("transaction_successful"))
    .count()
)
success_rate = (
    successful_transactions / total_transactions * 100
)

kpi_success_rate = spark.createDataFrame(
    [(total_transactions, success_rate)],
    ["Successful_Rows", "Success_Rate_Pct"]
)

display(kpi_success_rate)
# print(f"Transaction Success Rate: {success_rate:.2f}%")

## 14. KPI 9 — Missing Data Rate
Two versions, since "missing data rate" is ambiguous:
- **Cell-level**: nulls across all cells (post sentinel-normalization) ÷ total cells.
- **Column-level breakdown**: null rate per column, so you can see which fields are actually driving it.

In [0]:
business_cols = [c for c in raw.columns]  # original source columns only, excludes derived flag columns

null_counts = cleaned.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in business_cols]
).first().asDict()

total_cells = TOTAL_ROWS * len(business_cols)
total_missing_cells = sum(null_counts.values())
cell_level_missing_rate = round(total_missing_cells / total_cells * 100, 2) if total_cells else None

print(f"Cell-level Missing Data Rate: {cell_level_missing_rate}%")

kpi_missing_by_column = spark.createDataFrame(
    [(col, cnt, round(cnt / TOTAL_ROWS * 100, 2)) for col, cnt in null_counts.items()],
    ["Column", "Null_Count", "Null_Rate_Pct"]
).orderBy(F.desc("Null_Rate_Pct"))

display(kpi_missing_by_column)

## 15. KPI 10 — Invalid Date Percentage
Split into two numbers on purpose: "invalid" (non-null string that failed to parse) is a different
failure mode than "missing" (null outright). Reported both individually and combined.

In [0]:
invalid_date_count = cleaned.filter(F.col("is_invalid_date")).count()
missing_date_count = cleaned.filter(F.col("date_is_missing")).count()

invalid_date_pct = round(invalid_date_count / TOTAL_ROWS * 100, 2) if TOTAL_ROWS else None
missing_date_pct = round(missing_date_count / TOTAL_ROWS * 100, 2) if TOTAL_ROWS else None
combined_bad_date_pct = round((invalid_date_count + missing_date_count) / TOTAL_ROWS * 100, 2) if TOTAL_ROWS else None

print(f"Invalid (unparseable) date %: {invalid_date_pct}%")
print(f"Missing (null) date %: {missing_date_pct}%")
print(f"Combined bad-date %: {combined_bad_date_pct}%")

kpi_invalid_dates = spark.createDataFrame(
    [(invalid_date_count, invalid_date_pct, missing_date_count, missing_date_pct, combined_bad_date_pct)],
    ["Invalid_Count", "Invalid_Pct", "Missing_Count", "Missing_Pct", "Combined_Bad_Date_Pct"]
)
display(kpi_invalid_dates)